# Zero Density in Symmetric-Group Character Tables

This notebook plots the density of zero entries in character tables of `S_n`.

- Small values of `n` are computed directly with `murnaghan_nakayama.get_character_table`.
- The `S_40` value is computed by streaming `runs/S40/S40.csv` or `runs/S40/S40.csv.gz`, so the full 4.6 GB table is never loaded into memory.
- Zero density means `number of zero entries / total number of entries`.

In [ ]:
from pathlib import Path
import csv
import gzip
import time

import matplotlib.pyplot as plt
import numpy as np

import murnaghan_nakayama as mn

## Configuration

Increase `SMALL_N_MAX` if your machine can compute more direct tables. The default computes through `S_14`, which is still small enough for ordinary laptops. The notebook looks for the large `S_40` file directly under the repository root first, matching the VM layout you described.


In [ ]:
SMALL_N_MAX = 25
INCLUDE_S40 = True
S40_CANDIDATES = [
    Path("S40.csv.gz"),
    Path("S40.csv"),
    Path("runs/S40/S40.csv.gz"),
    Path("runs/S40/S40.csv"),
]
CSV_PROGRESS_INTERVAL = 1000
HEATMAP_BINS = 800
HEATMAP_PROGRESS_INTERVAL = 1000


## Helper functions

In [ ]:
def open_text_maybe_gzip(path):
    if str(path).endswith(".gz"):
        return gzip.open(path, "rt", newline="")
    return open(path, "r", newline="")


def zero_density_array(table):
    zero_count = int(np.count_nonzero(table == 0))
    total_count = int(table.size)
    return {
        "zero_count": zero_count,
        "total_count": total_count,
        "density": zero_count / total_count,
    }


def zero_density_csv(path, expected_size=None, progress_interval=1000):
    path = Path(path)
    zero_count = 0
    total_count = 0
    rows = 0
    width = None
    start = time.perf_counter()

    with open_text_maybe_gzip(path) as file:
        reader = csv.reader(file)
        for row in reader:
            rows += 1
            if width is None:
                width = len(row)
            elif len(row) != width:
                raise ValueError(
                    f"Ragged row {rows}: expected {width} columns, found {len(row)}"
                )

            total_count += len(row)
            zero_count += sum(1 for value in row if value.strip() == "0")

            if progress_interval and rows % progress_interval == 0:
                elapsed = time.perf_counter() - start
                if expected_size:
                    percent = 100 * rows / expected_size
                    print(
                        f"{path.name}: {rows}/{expected_size} rows "
                        f"({percent:.1f}%), elapsed {elapsed:.1f}s"
                    )
                else:
                    print(f"{path.name}: {rows} rows, elapsed {elapsed:.1f}s")

    if expected_size is not None and rows != expected_size:
        raise ValueError(f"Expected {expected_size} rows, found {rows}")
    if expected_size is not None and width != expected_size:
        raise ValueError(f"Expected {expected_size} columns, found {width}")

    elapsed = time.perf_counter() - start
    return {
        "zero_count": zero_count,
        "total_count": total_count,
        "density": zero_count / total_count,
        "rows": rows,
        "columns": width,
        "elapsed_seconds": elapsed,
        "path": str(path),
    }


def find_first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return None

def zero_heatmap_blocks_csv(path, expected_size, bins=800, progress_interval=1000):
    path = Path(path)
    zero_counts = np.zeros((bins, bins), dtype=np.uint64)
    row_bin_sizes = np.zeros(bins, dtype=np.uint64)
    col_bin_sizes = np.zeros(bins, dtype=np.uint64)
    col_bins = np.array(
        [min(col * bins // expected_size, bins - 1) for col in range(expected_size)],
        dtype=np.int32,
    )
    for col_bin in col_bins:
        col_bin_sizes[col_bin] += 1

    rows = 0
    width = None
    start = time.perf_counter()
    with open_text_maybe_gzip(path) as file:
        reader = csv.reader(file)
        for row in reader:
            rows += 1
            if width is None:
                width = len(row)
                if width != expected_size:
                    raise ValueError(f"Expected {expected_size} columns, found {width}")
            elif len(row) != width:
                raise ValueError(
                    f"Ragged row {rows}: expected {width} columns, found {len(row)}"
                )

            row_bin = min((rows - 1) * bins // expected_size, bins - 1)
            row_bin_sizes[row_bin] += 1
            for value, col_bin in zip(row, col_bins):
                if value.strip() == "0":
                    zero_counts[row_bin, col_bin] += 1

            if progress_interval and rows % progress_interval == 0:
                elapsed = time.perf_counter() - start
                percent = 100 * rows / expected_size
                print(
                    f"{path.name} heatmap: {rows}/{expected_size} rows "
                    f"({percent:.1f}%), elapsed {elapsed:.1f}s"
                )

    if rows != expected_size:
        raise ValueError(f"Expected {expected_size} rows, found {rows}")

    block_sizes = np.outer(row_bin_sizes, col_bin_sizes)
    zero_density = np.divide(
        zero_counts,
        block_sizes,
        out=np.zeros_like(zero_counts, dtype=np.float64),
        where=block_sizes != 0,
    )
    elapsed = time.perf_counter() - start
    return {
        "zero_density": zero_density,
        "zero_counts": zero_counts,
        "block_sizes": block_sizes,
        "rows": rows,
        "columns": width,
        "elapsed_seconds": elapsed,
        "path": str(path),
        "bins": bins,
    }


## Compute densities for directly computed `n`

These tables are computed directly. With `SMALL_N_MAX = 14`, this gives a denser set of points before the streamed `S_40` value. The row and column ordering does not affect zero density.


In [ ]:
results = []

for n in range(1, SMALL_N_MAX + 1):
    start = time.perf_counter()
    mn.clear_caches()
    table = mn.get_character_table(n, memo_file_name="")
    stats = zero_density_array(table)
    elapsed = time.perf_counter() - start
    results.append({"n": n, "source": "computed", "elapsed_seconds": elapsed, **stats})
    print(
        f"S_{n}: p(n)={table.shape[0]}, zeros={stats['zero_count']}/"
        f"{stats['total_count']}, density={stats['density']:.6f}, elapsed={elapsed:.2f}s"
    )

## Stream the `S_40` table

This cell reads the large CSV one row at a time. It supports either the raw CSV or the gzip-compressed CSV.

In [ ]:
if INCLUDE_S40:
    s40_path = find_first_existing(S40_CANDIDATES)
    if s40_path is None:
        print("Skipping S_40: no file found at", [str(path) for path in S40_CANDIDATES])
    else:
        expected_size = len(list(mn.partitions(40)))
        print(f"Streaming {s40_path} with expected shape {expected_size} x {expected_size}")
        stats = zero_density_csv(
            s40_path,
            expected_size=expected_size,
            progress_interval=CSV_PROGRESS_INTERVAL,
        )
        results = [result for result in results if result["n"] != 40]
        results.append({"n": 40, "source": "csv", **stats})
        print(
            f"S_40: zeros={stats['zero_count']}/{stats['total_count']}, "
            f"density={stats['density']:.6f}, elapsed={stats['elapsed_seconds']:.2f}s"
        )

## Plot zero density

In [ ]:
plot_results = sorted(results, key=lambda item: item["n"])
xs = [item["n"] for item in plot_results]
ys = [item["density"] for item in plot_results]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(xs, ys, marker="o", linewidth=2)
ax.set_xlabel("n")
ax.set_ylabel("Density of zero entries")
ax.set_title("Zero density in character tables of $S_n$")
ax.set_ylim(0, max(ys) * 1.1 if ys else 1)
ax.grid(True, alpha=0.3)

for item in plot_results:
    label = "S40 CSV" if item["n"] == 40 and item.get("source") == "csv" else None
    if label:
        ax.annotate(label, (item["n"], item["density"]), xytext=(6, 6), textcoords="offset points")

plt.show()

## S_40 zero-pattern heatmap

This is a block-downsampled heatmap of the zero pattern. Each pixel shows the fraction of entries equal to zero in a block of the full `S_40` table. Bright cells are zero-heavy. This avoids storing a literal 37,338 by 37,338 image in memory.


In [ ]:
s40_heatmap = None
if INCLUDE_S40:
    s40_path = find_first_existing(S40_CANDIDATES)
    if s40_path is None:
        print("Skipping S_40 heatmap: no file found at", [str(path) for path in S40_CANDIDATES])
    else:
        expected_size = len(list(mn.partitions(40)))
        print(
            f"Building {HEATMAP_BINS} x {HEATMAP_BINS} zero-density heatmap "
            f"from {s40_path}"
        )
        s40_heatmap = zero_heatmap_blocks_csv(
            s40_path,
            expected_size=expected_size,
            bins=HEATMAP_BINS,
            progress_interval=HEATMAP_PROGRESS_INTERVAL,
        )
        print(
            f"Finished heatmap in {s40_heatmap['elapsed_seconds']:.2f}s; "
            f"shape={s40_heatmap['zero_density'].shape}"
        )


## Plot S_40 zero-pattern heatmap


In [ ]:
if s40_heatmap is None:
    print("Run the heatmap cell above after placing S40.csv or S40.csv.gz in the repo root.")
else:
    fig, ax = plt.subplots(figsize=(9, 8))
    image = ax.imshow(
        s40_heatmap["zero_density"],
        origin="upper",
        interpolation="nearest",
        cmap="magma",
        vmin=0,
        vmax=1,
    )
    ax.set_title("Zero pattern heatmap for $S_{40}$")
    ax.set_xlabel("Conjugacy class index block")
    ax.set_ylabel("Irreducible representation index block")
    colorbar = fig.colorbar(image, ax=ax)
    colorbar.set_label("Fraction of entries equal to 0")
    plt.tight_layout()
    plt.show()


## Data table

In [ ]:
for item in plot_results:
    print(
        f"n={item['n']:>2} source={item['source']:<8} "
        f"zero_density={item['density']:.8f} "
        f"zeros={item['zero_count']} total={item['total_count']}"
    )